## Structured Output

MOdel can be requested to provide their response in a formate matching a given Schema. This is usefull for ensuring the output can be easily parsed and used in subsequent processing. LangChain support multiple schema types and method for enforcing structured output.


### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions and nested structures.

In [4]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain.chat_models import init_chat_model

# Load environment variables from .env
load_dotenv(find_dotenv(), override=True)

# Initialize Groq chat model
model = init_chat_model("groq:qwen/qwen3.8-27b",  max_tokens=500)
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, client=<groq.resources.chat.completions.Completions object at 0x0000021DF61044C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021DF60DBE80>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=500)

In [5]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="The year the movie was released")
    director: str=Field(description="The Director of the movie")
    rating: float=Field(description="The movie rating out of 10")
    


In [6]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, client=<groq.resources.chat.completions.Completions object at 0x0000021DF61044C0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000021DF60DBE80>, model_name='qwen/qwen3.8-27b', model_kwargs={}, groq_api_key=SecretStr('**********'), max_tokens=500), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The Director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', 'year', 'director', 'rating'], 'type': 'object'}}}], 'ls_structured_output_format': {'kwargs': {'method': 'function_calling'}, 'schema': {'type': 'function', 'function'

In [7]:
model.invoke("Provide the details about the movie Inception")

AIMessage(content='**Inception** is a 2010 science fiction action thriller film written and directed by **Christopher Nolan**. It is widely regarded as one of the most complex and iconic films of the 21st century, blending heist movie tropes with high-concept science fiction.\n\n### Basic Information\n- **Director/Writer:** Christopher Nolan\n- **Release Date:** July 16, 2010 (US)\n- **Runtime:** 148 minutes\n- **Genre:** Sci-Fi, Action, Thriller, Drama\n- **Studio:** Warner Bros. Pictures\n- **Music:** Hans Zimmer\n- **Box Office:** ~$836 million worldwide (one of the highest-grossing films of its year)\n\n### Cast\n- **Leonardo DiCaprio** as Dom Cobb\n- **Joseph Gordon-Levitt** as Arthur\n- **Elliot Page** as Ariadne\n- **Tom Hardy** as Eames\n- **Ken Watanabe** as Saito\n- **Marion Cotillard** as Mal\n- **Cillian Murphy** as Robert Fischer\n- **Michael Caine** as Miles\n- **Tom Berenger** as Browning\n\n### Plot Summary\nThe film revolves around **Dom Cobb** (Leonardo DiCaprio), a s

In [8]:
model_with_structure.invoke("Provide the details about the movie Inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Message output alongside parsed structure

In [9]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title: str=Field(description="The title of the movie")
    year: int=Field(description="The year the movie was released")
    director: str=Field(description="The Director of the movie")
    rating: float=Field(description="The movie rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response=model_with_structure.invoke("Provide the details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'ywjx6nm5w', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 68, 'prompt_tokens': 350, 'total_tokens': 418, 'completion_time': 0.170011216, 'completion_tokens_details': None, 'prompt_time': 0.025453877, 'prompt_tokens_details': None, 'queue_time': 0.069801701, 'total_time': 0.195465093}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fd97-3179-77f3-9cc2-d7c5efbe334d-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Christopher Nolan', 'rating': 8.8, 'title': 'Inception', 'year': 2010}, 'id': 'ywjx6nm5w', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 350, 'output_tokens': 68, 'total

### Nested Structure

In [10]:
class Actor(BaseModel):
    name:str
    role:str

class Moviedetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budge: float | None= Field(None, description="Budget in million USD")

In [11]:

model_with_structure = model.with_structured_output(Moviedetails, include_raw=True)
response=model_with_structure.invoke("Provide the details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'q3xmym8kn', 'function': {'arguments': '{"cast":[{"name":"Leonardo DiCaprio","role":"Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Elliot Page","role":"Ariadne"},{"name":"Tom Hardy","role":"Eames"},{"name":"Ken Watanabe","role":"Saito"},{"name":"Marion Cotillard","role":"Mal"}],"genres":["Action","Sci-Fi","Thriller"],"title":"Inception","year":2010}', 'name': 'Moviedetails'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 173, 'prompt_tokens': 392, 'total_tokens': 565, 'completion_time': 0.393476131, 'completion_tokens_details': None, 'prompt_time': 0.032504708, 'prompt_tokens_details': None, 'queue_time': 0.06013304, 'total_time': 0.425980839}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_4560dae850', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fd97-32d7-78a1-a8ab-62b130e3fa2

### TypedDict

TypeDict provide a simpler alternative using python's built-in typing, ideal when you don't need runtime validation.

In [12]:
from typing_extensions import TypedDict, Annotated

In [13]:
class MovieDict(TypedDict):
    """A Movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the Movie was relesed "]
    director: Annotated[str, ..., "The Director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [14]:
model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [15]:
class Actor(TypedDict):
    name:str
    role:str

class Moviedetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget: float | None= Field(None, description="Budget in million USD")

model_withtypedict=model.with_structured_output(Moviedetails)
response=model_withtypedict.invoke("Please provide the details of the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Iron Man / Tony Stark'},
  {'name': 'Chris Evans', 'role': 'Captain America / Steve Rogers'},
  {'name': 'Scarlett Johansson', 'role': 'Black Widow / Natasha Romanoff'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Mark Ruffalo', 'role': 'Hulk / Bruce Banner'},
  {'name': 'Jeremy Renner', 'role': 'Hawkeye / Clint Barton'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Paul Bettany', 'role': 'Vision / J.A.R.V.I.S.'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'}],
 'genres': ['Action', 'Adventure', 'Science Fiction'],
 'title': 'Avengers',
 'year': 2012}

In [16]:
model.profile

## Data Classes

A data class is a class typically containing mainly data, although there aren't really any reactions. You create it using the @dataclass decorator.

In [17]:
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent =create_agent(
    model = init_chat_model("groq:qwen/qwen3.8-27b",  max_tokens=500),
    response_format=ContactInfo
)

result= agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='98fa0f34-38f2-4051-95d1-69342cfeab6c'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '42cn0td0a', 'function': {'arguments': '{"email":"john@example.com","name":"John Doe","phone":"(555) 123-4567"}', 'name': 'ContactInfo'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 65, 'prompt_tokens': 351, 'total_tokens': 416, 'completion_time': 0.211921194, 'completion_tokens_details': None, 'prompt_time': 0.024720897, 'prompt_tokens_details': None, 'queue_time': 0.053440811, 'total_time': 0.236642091}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_21e59ac2de', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a0fd97-4982-7be3-96d6-732a896ca096-0', tool_calls=[{'name': 'ContactInfo', 'args': {'email': 'john@exa

In [18]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [19]:
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent =create_agent(
    model = init_chat_model("groq:qwen/qwen3.8-27b",  max_tokens=500),
    response_format=ContactInfo
)

result= agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [20]:
# Dataclass

from dataclasses import dataclass

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

agent =create_agent(
    model = init_chat_model("groq:qwen/qwen3.8-27b",  max_tokens=500),
    response_format=ContactInfo
)

result= agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]


ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')